In [ ]:
using Pkg
Pkg.activate(".")
Pkg.develop(path="..")

using Revise

In [ ]:
if success(`nvidia-smi`)
    println("CUDA is available. Loading CUDA.jl...")
    using CUDA
end

In [ ]:
grid =  bslLD.Grid([0.0,-4.0,-4.0],[60.0,4.0,4.0],[64,33,33],0.02,20000,1, 1.0, 3)

# initFuncv(v)= exp(-(v+2)^2 / 2) / sqrt(2*pi)+ exp(-(v-2)^2 / 2) / sqrt(2*pi)
initFuncv(v) = exp(-v^2 / 2) / sqrt(2*pi)
initFuncx(x) = 1+ 0.000001 * rand()
f = bslLD.Distribution(grid, 0.5,initFuncv=initFuncv, initFuncx=initFuncx);
e = bslLD.empty_vectorfield(grid);

In [ ]:
mutable struct Diag
    rho::Vector
    f::Vector
    Ex::Vector
end
Diag() = Diag([], [], [])

function diags!(diags, f, rho, Ex, grid)
    grid.index[1] % 1 == 0 || return
    push!(diags.f, copy(f.data .- mean(f.data, dims=1)))
    push!(diags.rho, copy(rho.data[:]))
    push!(diags.Ex, copy(Ex.data[:]))
end

function step!(f, grid, diags)
    bslLD.advectX!(f, grid)
    rho = bslLD.compute_density(f, grid)
    sol = bslLD.solve_fields(bslLD.Moments(rho), grid, bslLD.AdiabaticFieldSolver())
    bslLD.advectV!(f, grid, sol.E)
    diags!(diags, f, rho, sol.E[1], grid)
end


In [ ]:
f.data

In [ ]:
diags = Diag()
for i in grid.itime
    grid.index[1] = i
    step!(f, grid, diags)
end


In [ ]:
plot(grid.xaxes[1],diags.rho, label="rho", legend=:none)

In [ ]:
using FFTW, DSP


In [ ]:
locData = transpose(hcat(map(x-> x.-mean(x), Array.(diags.rho))...))


Nx, Ny = size(locData)
w = kaiser(Ny, 6)

windowed = locData .* w'        # broadcast along second dim (1 × Ny)

heatmap(log.(abs.(fft(windowed))[1:400,1:round(Int,Ny/2)]))

In [ ]:
plot(map(x->(mean((x.-mean(x)).^2)), Array.(rhodiag)), yscale=:log10)